# 06_construct_robustness

Build a pooled chief-complaint component lookup table for radiology, pressor, and vent.

This notebook:
- reads the cleaned analytic files under `./main/shared/...`
- pools comma-split chief complaint components across all three datasets
- applies light normalization for deduplication
- skips `no_cc`
- labels each pooled unique normalized component with a local LLM
- writes results directly to `./main/chief_complaint_lookup_table.jsonl`
- writes failures to `./main/chief_complaint_lookup_table_errors.jsonl`


In [16]:
from pathlib import Path
from typing import Literal
import json
import re
import time

import pandas as pd
import requests
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
from IPython.display import display

ROOT = Path(".")
MAIN = ROOT / "main"
SHARED = MAIN / "shared"

RAD_PATH = SHARED / "radiology" / "radiology_analytic_imputed.parquet"
PRESSOR_PATH = SHARED / "pressor" / "pressor_analytic_imputed.parquet"
VENT_PATH = SHARED / "vent" / "vent_analytic_imputed.parquet"

TEXT_COL = "chiefcomplaint"
NO_CC_TOKEN = "no_cc"

OUTPUT_JSONL = MAIN / "chief_complaint_lookup_table.jsonl"
ERROR_JSONL = MAIN / "chief_complaint_lookup_table_errors.jsonl"

DATASET_PATHS = {
    "radiology": RAD_PATH,
    "pressor": PRESSOR_PATH,
    "vent": VENT_PATH,
}

MAIN.mkdir(parents=True, exist_ok=True)
print("Output jsonl:", OUTPUT_JSONL)
print("Error jsonl:", ERROR_JSONL)


Output jsonl: main/chief_complaint_lookup_table.jsonl
Error jsonl: main/chief_complaint_lookup_table_errors.jsonl


In [17]:
frames = {}

for name, path in DATASET_PATHS.items():
    df = pd.read_parquet(path).copy()
    if TEXT_COL not in df.columns:
        raise KeyError(f"{TEXT_COL} not found in {path}")
    frames[name] = df
    print(f"{name}: {path} -> {df.shape}")

print("Chief complaint column:", TEXT_COL)


radiology: main/shared/radiology/radiology_analytic_imputed.parquet -> (203016, 122)
pressor: main/shared/pressor/pressor_analytic_imputed.parquet -> (69223, 66)
vent: main/shared/vent/vent_analytic_imputed.parquet -> (34042, 66)
Chief complaint column: chiefcomplaint


In [18]:
ABBREVIATION_PATTERNS = [
    (r"\bn\s*/\s*v\s*/\s*d\b", "nausea vomiting diarrhea"),
    (r"\bn\s*/\s*v\b", "nausea vomiting"),
    (r"\bs\s*/\s*p\b", "status post"),
    (r"^sp\b", "status post"),
    (r"\bsob\b", "shortness of breath"),
    (r"\bcp\b", "chest pain"),
    (r"\babd\b", "abdominal"),
    (r"\bams\b", "altered mental status"),
    (r"\bmvc\b", "motor vehicle collision"),
]

def normalize_component(text: str) -> str:
    x = str(text).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"\s*/\s*", "/", x)

    for pattern, replacement in ABBREVIATION_PATTERNS:
        x = re.sub(pattern, replacement, x)

    x = re.sub(r"\s+", " ", x).strip()
    return x

def extract_component_records(series: pd.Series, dataset_name: str) -> pd.DataFrame:
    rows = []
    s = series.astype("string")

    for raw in s.dropna():
        text = str(raw)
        if text.strip().lower() == NO_CC_TOKEN:
            continue

        for part in text.split(","):
            raw_component = part.strip()
            if raw_component == "":
                continue
            if raw_component.lower() == NO_CC_TOKEN:
                continue

            normalized_component = normalize_component(raw_component)
            if normalized_component == "" or normalized_component == NO_CC_TOKEN:
                continue

            rows.append(
                {
                    "dataset": dataset_name,
                    "raw_component": raw_component,
                    "normalized_component": normalized_component,
                }
            )

    return pd.DataFrame(rows)

def build_component_table(frame_map: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, pd.DataFrame]:
    parts = []
    for name, df in frame_map.items():
        parts.append(extract_component_records(df[TEXT_COL], dataset_name=name))

    all_components = pd.concat(parts, ignore_index=True)

    dataset_counts = (
        all_components.groupby(["normalized_component", "dataset"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for name in frame_map:
        if name not in dataset_counts.columns:
            dataset_counts[name] = 0

    dataset_counts = dataset_counts.rename(columns={name: f"n_{name}" for name in frame_map})

    raw_variant_counts = (
        all_components.groupby(["normalized_component", "raw_component"])
        .size()
        .rename("n_raw")
        .reset_index()
        .sort_values(["normalized_component", "n_raw", "raw_component"], ascending=[True, False, True])
        .reset_index(drop=True)
    )

    raw_summary = (
        raw_variant_counts.groupby("normalized_component")
        .agg(
            component_example=("raw_component", "first"),
            n_raw_variants=("raw_component", "nunique"),
            raw_examples_top5=("raw_component", lambda x: " | ".join(x.head(5))),
        )
        .reset_index()
    )

    out = dataset_counts.merge(raw_summary, on="normalized_component", how="left")

    count_cols = [f"n_{name}" for name in frame_map]
    out["n_total"] = out[count_cols].sum(axis=1)
    out["n_datasets_present"] = (out[count_cols] > 0).sum(axis=1)

    out = out.sort_values(
        ["n_total", "n_datasets_present", "normalized_component"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    ordered_cols = [
        "normalized_component",
        "component_example",
        "raw_examples_top5",
        "n_raw_variants",
        "n_total",
        "n_datasets_present",
        "n_radiology",
        "n_pressor",
        "n_vent",
    ]
    return out.loc[:, ordered_cols], all_components

component_table, component_records = build_component_table(frames)

print("Unique normalized components:", len(component_table))
display(component_table.head(20))


Unique normalized components: 13958


,normalized_component,component_example,raw_examples_top5,n_raw_variants,n_total,n_datasets_present,n_radiology,n_pressor,n_vent
0,transfer,Transfer,Transfer | TRANSFER,2,28257,3,20287,5067,2903
1,abdominal pain,Abd pain,Abd pain | ABD PAIN | ABDOMINAL PAIN | ABD PA...,8,21228,3,18996,1498,734
2,dyspnea,Dyspnea,Dyspnea | DYSPNEA,2,18080,3,13573,2691,1816
3,chest pain,Chest pain,Chest pain | CHEST PAIN | CP | CHEST PAIN | c...,6,17749,3,15920,1289,540
4,status post fall,s/p Fall,s/p Fall | S/P FALL | SP FALL | S/P FALL,4,12551,3,10746,1187,618
5,fever,Fever,Fever | FEVER | fever,3,10788,3,9062,1204,522
6,weakness,Weakness,Weakness | WEAKNESS | weakness,3,8365,3,6899,973,493
7,si,SI,SI,1,7209,3,7136,53,20
8,altered mental status,Altered mental status,Altered mental status | ALTERED MENTAL STATUS ...,3,7003,3,5123,1245,635
9,etoh,ETOH,ETOH | EtOH | Etoh | etoh,4,5726,3,5552,116,58


In [19]:
class ChiefComplaintComponentLabel(BaseModel):
    is_informative: bool = Field(
        description="False if the component is too vague, truncated, malformed, directional only, or otherwise not clinically interpretable on its own."
    )
    canonical_label: str = Field(
        description="Short English label. Use 'not_informative' when is_informative is false."
    )
    trauma_status: Literal["trauma", "non_trauma", "uncertain", "not_informative"]
    body_system: Literal[
        "cardiac",
        "respiratory",
        "neurologic",
        "gastrointestinal",
        "genitourinary",
        "obstetric_gynecologic",
        "psychiatric",
        "musculoskeletal",
        "skin_soft_tissue",
        "constitutional",
        "endocrine_metabolic",
        "hematologic",
        "infectious",
        "nonspecific_other",
        "not_informative",
    ]
    symptom_type: Literal[
        "symptom",
        "sign_or_objective_finding",
        "injury",
        "procedure_or_device",
        "administrative_or_context",
        "uncertain",
        "not_informative",
    ]
    acuity_1to5: int = Field(ge=0, le=5, description="0 only when is_informative is false.")
    severity_1to5: int = Field(ge=0, le=5, description="0 only when is_informative is false.")
    specificity_1to5: int = Field(ge=0, le=5, description="0 only when is_informative is false.")
    red_flag: bool
    objective_finding: bool
    note: str = Field(description="One short sentence.")

SCHEMA = ChiefComplaintComponentLabel.model_json_schema()

SYSTEM_PROMPT = """
You are labeling one chief complaint component from hospital EHR data.

Context:
- The text comes from the chief complaint field.
- It is a short complaint fragment, not a full sentence.
- It may contain common clinical abbreviations used in emergency or ICU settings, such as SOB, CP, abd pain, N/V, AMS, or MVC.

Tasks:
1. Decide whether the component is informative for clinical presentation.
2. If it is informative, extract:
   - canonical_label
   - trauma_status
   - body_system
   - symptom_type
   - acuity_1to5
   - severity_1to5
   - specificity_1to5
   - red_flag
   - objective_finding
   - note
3. If the component is too vague, truncated, malformed, directional only, or otherwise not clinically interpretable by itself, set is_informative=false and use:
   - canonical_label = "not_informative"
   - trauma_status = "not_informative"
   - body_system = "not_informative"
   - symptom_type = "not_informative"
   - acuity_1to5 = 0
   - severity_1to5 = 0
   - specificity_1to5 = 0
   - red_flag = false
   - objective_finding = false

Scoring guidance:
- acuity_1to5: immediate urgency at presentation
- severity_1to5: likely clinical seriousness if the complaint is real
- specificity_1to5: how clinically specific and informative the phrase is on its own

Return only valid JSON.
"""


In [20]:
MODEL = "gpt-oss:20b"
OLLAMA_CHAT_URL = "http://localhost:11434/api/chat"

import requests
import json

REQUEST_TIMEOUT = (10, 10)  # (connect timeout, read timeout) in seconds

def build_user_message(row: pd.Series) -> str:
    return "\n".join([
        f"normalized_component: {row['normalized_component']}",
        f"component_example: {row['component_example']}",
        f"raw_examples_top5: {row['raw_examples_top5']}",
    ])

def classify_one_row(row: pd.Series) -> dict:
    user_msg = build_user_message(row)

    payload = {
        "model": MODEL,
        "stream": False,
        "format": SCHEMA,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        "options": {
            "temperature": 0
        }
    }

    response = requests.post(
        OLLAMA_CHAT_URL,
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()

    out = response.json()
    raw = out["message"]["content"]
    obj = json.loads(raw)
    result = ChiefComplaintComponentLabel.model_validate(obj)

    return {
        "is_informative": bool(result.is_informative),
        "canonical_label": result.canonical_label.strip(),
        "trauma_status": str(result.trauma_status).strip(),
        "body_system": str(result.body_system).strip(),
        "symptom_type": str(result.symptom_type).strip(),
        "acuity_1to5": int(result.acuity_1to5),
        "severity_1to5": int(result.severity_1to5),
        "specificity_1to5": int(result.specificity_1to5),
        "red_flag": bool(result.red_flag),
        "objective_finding": bool(result.objective_finding),
        "note": result.note.strip(),
    }

In [21]:
# Single-row sanity check

i = 0
sample_out = classify_one_row(component_table.iloc[i])
sample_out


{'is_informative': False,
 'canonical_label': 'not_informative',
 'trauma_status': 'not_informative',
 'body_system': 'not_informative',
 'symptom_type': 'not_informative',
 'acuity_1to5': 0,
 'severity_1to5': 0,
 'specificity_1to5': 0,
 'red_flag': False,
 'objective_finding': False,
 'note': "The component 'Transfer' indicates a patient transfer status rather than a clinical symptom or complaint."}

In [22]:
def append_jsonl(path: Path, record: dict) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def load_existing_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()

    rows = []
    bad_lines = 0

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad_lines += 1

    if bad_lines > 0:
        print(f"Skipped malformed lines in {path.name}: {bad_lines}")

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)

def run_lookup_resumable(
    component_table: pd.DataFrame,
    jsonl_path: Path = OUTPUT_JSONL,
    error_jsonl_path: Path = ERROR_JSONL,
    sleep_seconds: float = 0.0,
):
    existing_df = load_existing_jsonl(jsonl_path)
    done_ids = set()

    if not existing_df.empty and "normalized_component" in existing_df.columns:
        done_ids = set(existing_df["normalized_component"].astype(str).tolist())

    pending_df = component_table.copy()
    pending_df["normalized_component"] = pending_df["normalized_component"].astype(str)
    pending_df = pending_df[~pending_df["normalized_component"].isin(done_ids)].reset_index(drop=True)

    print(f"existing completed rows: {len(done_ids)}")
    print(f"pending rows: {len(pending_df)}")

    pbar = tqdm(pending_df.iterrows(), total=len(pending_df))

    for _, row in pbar:
        row_dict = row.to_dict()
        current_key = str(row_dict["normalized_component"])
        pbar.set_postfix_str(current_key)

        try:
            ann = classify_one_row(row)
            record = {**row_dict, **ann}
            append_jsonl(jsonl_path, record)

            if sleep_seconds > 0:
                time.sleep(sleep_seconds)

        except Exception as e:
            print(f"\nFAILED: {current_key} | {type(e).__name__}: {e}")
            error_record = {
                **row_dict,
                "error_type": type(e).__name__,
                "error_message": str(e),
            }
            append_jsonl(error_jsonl_path, error_record)

    final_df = load_existing_jsonl(jsonl_path)

    print("done")
    print(f"jsonl saved to: {jsonl_path}")
    print(f"errors saved to: {error_jsonl_path}")

    return final_df

In [23]:
lookup_table = run_lookup_resumable(component_table=component_table)
display(lookup_table.head(20))


Skipped malformed lines in chief_complaint_lookup_table.jsonl: 1
existing completed rows: 13945
pending rows: 13


  8%|▊         | 1/13 [00:10<02:00, 10.00s/it, boerrhaves]


FAILED: b rib pain | ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=10)


 15%|█▌        | 2/13 [00:20<01:50, 10.00s/it, dt"s si]   


FAILED: boerrhaves | ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=10)


...

 92%|█████████▏| 12/13 [02:00<00:10, 10.01s/it, status post sz ^ammonia level]


FAILED: status post palps | ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=10)


100%|██████████| 13/13 [02:10<00:00, 10.01s/it, status post sz ^ammonia level]


FAILED: status post sz ^ammonia level | ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=10)
Skipped malformed lines in chief_complaint_lookup_table.jsonl: 1
done
jsonl saved to: main/chief_complaint_lookup_table.jsonl
errors saved to: main/chief_complaint_lookup_table_errors.jsonl


,normalized_component,component,component_examples,n_total,n_datasets_present,n_radiology,n_pressor,n_vent,is_informative,canonical_label,...,symptom_type,acuity_1to5,severity_1to5,specificity_1to5,red_flag,objective_finding,note,component_example,raw_examples_top5,n_raw_variants
0,transfer,TRANSFER,"[TRANSFER, Transfer]",28257,3,20287,5067,2903,False,not_informative,...,not_informative,0,0,0,False,False,The term 'transfer' alone does not provide a c...,NaN,NaN,NaN
1,abdominal pain,ABD PAIN,"[ABD PAIN, ABD PAIN, ABD PAIN, ABDOMINAL PA...",21228,3,18996,1498,734,True,abdominal pain,...,procedure_or_device,3,3,2,False,False,Generic abdominal pain complaint; no additiona...,NaN,NaN,NaN
2,dyspnea,DYSPNEA,"[DYSPNEA, Dyspnea]",18080,3,13573,2691,1816,True,dyspnea,...,symptom,3,3,4,False,False,shortness of breath,NaN,NaN,NaN
3,chest pain,CHEST PAIN,"[CHEST PAIN, CHEST PAIN, CP, Chest pain, ches...",17749,3,15920,1289,540,True,Chest pain,...,procedure_or_device,4,4,3,True,False,Chest pain is a common symptom that may indica...,NaN,NaN,NaN
4,status post fall,S/P FALL,"[S/P FALL, S/P FALL, SP FALL, s/p Fall]",12551,3,10746,1187,618,True,post-fall,...,injury,3,3,3,False,False,"Patient is status post a fall, indicating pote...",NaN,NaN,NaN
5,fever,FEVER,"[FEVER, Fever, fever]",10788,3,9062,1204,522,True,fever,...,symptom,2,2,3,False,False,fever indicates a systemic response to infecti...,NaN,NaN,NaN
6,weakness,WEAKNESS,"[WEAKNESS, Weakness, weakness]",8365,3,6899,973,493,True,weakness,...,symptom,3,3,3,False,False,General weakness reported as a chief complaint.,NaN,NaN,NaN
7,si,SI,[SI],7209,3,7136,53,20,False,not_informative,...,not_informative,0,0,0,False,False,The component 'si' is too vague and not a reco...,NaN,NaN,NaN
8,altered mental status,ALTERED MENTAL STATUS,"[ALTERED MENTAL STATUS, AMS, Altered mental st...",7003,3,5123,1245,635,True,altered mental status,...,injury,5,5,3,True,False,AMS is a critical red flag indicating possible...,NaN,NaN,NaN
9,etoh,ETOH,"[ETOH, EtOH, Etoh, etoh]",5726,3,5552,116,58,True,alcohol intoxication,...,injury,3,3,4,True,False,"etoh indicates alcohol intoxication, often pre...",NaN,NaN,NaN


In [25]:
from json import JSONDecoder
import json
from pathlib import Path
import pandas as pd

output_path = OUTPUT_JSONL if "OUTPUT_JSONL" in globals() else Path("./main/chief_complaint_lookup_table.jsonl")
error_path = ERROR_JSONL if "ERROR_JSONL" in globals() else Path("./main/chief_complaint_lookup_table_errors.jsonl")

backup_main = output_path.with_name(output_path.stem + "_backup_before_manual_schema_fix.jsonl")
backup_err = error_path.with_name(error_path.stem + "_backup_before_manual_schema_fix.jsonl")

target_cols = [
    "normalized_component",
    "component",
    "component_examples",
    "n_total",
    "n_datasets_present",
    "n_radiology",
    "n_pressor",
    "n_vent",
    "is_informative",
    "canonical_label",
    "trauma_status",
    "body_system",
    "symptom_type",
    "acuity_1to5",
    "severity_1to5",
    "specificity_1to5",
    "red_flag",
    "objective_finding",
    "note",
]

patch_records = [
    {
        "normalized_component": "iph",
        "is_informative": True,
        "canonical_label": "intraparenchymal hemorrhage",
        "trauma_status": "non_trauma",
        "body_system": "neurologic",
        "symptom_type": "symptom",
        "acuity_1to5": 5,
        "severity_1to5": 5,
        "specificity_1to5": 4,
        "red_flag": True,
        "objective_finding": True,
        "note": "Likely chief complaint abbreviation for intraparenchymal hemorrhage (IPH); acute neurologic hemorrhagic presentation.",
    },
    {
        "normalized_component": "b rib pain",
        "is_informative": True,
        "canonical_label": "rib pain",
        "trauma_status": "non_trauma",
        "body_system": "musculoskeletal",
        "symptom_type": "symptom",
        "acuity_1to5": 3,
        "severity_1to5": 2,
        "specificity_1to5": 3,
        "red_flag": False,
        "objective_finding": False,
        "note": "Likely rib pain; leading abbreviation is ambiguous, possibly bilateral.",
    },
    {
        "normalized_component": "boerrhaves",
        "is_informative": True,
        "canonical_label": "boerhaave syndrome",
        "trauma_status": "non_trauma",
        "body_system": "gastrointestinal",
        "symptom_type": "symptom",
        "acuity_1to5": 5,
        "severity_1to5": 5,
        "specificity_1to5": 5,
        "red_flag": True,
        "objective_finding": True,
        "note": "Likely misspelling of Boerhaave syndrome.",
    },
    {
        "normalized_component": "dt\"s si",
        "is_informative": False,
        "canonical_label": "not_informative",
        "trauma_status": "not_informative",
        "body_system": "not_informative",
        "symptom_type": "not_informative",
        "acuity_1to5": 0,
        "severity_1to5": 0,
        "specificity_1to5": 0,
        "red_flag": False,
        "objective_finding": False,
        "note": "Ambiguous abbreviation string; not clinically interpretable on its own.",
    },
    {
        "normalized_component": "eu critial/motor vehicle collision",
        "is_informative": True,
        "canonical_label": "motor vehicle collision trauma",
        "trauma_status": "trauma",
        "body_system": "musculoskeletal",
        "symptom_type": "injury",
        "acuity_1to5": 4,
        "severity_1to5": 4,
        "specificity_1to5": 4,
        "red_flag": True,
        "objective_finding": False,
        "note": "Trauma mechanism is clear from motor vehicle collision; leading text is malformed.",
    },
    {
        "normalized_component": "ich-hip pain",
        "is_informative": True,
        "canonical_label": "hip pain",
        "trauma_status": "non_trauma",
        "body_system": "musculoskeletal",
        "symptom_type": "symptom",
        "acuity_1to5": 3,
        "severity_1to5": 2,
        "specificity_1to5": 3,
        "red_flag": False,
        "objective_finding": False,
        "note": "Hip pain is interpretable; leading ICH prefix is ambiguous.",
    },
    {
        "normalized_component": "left pta",
        "is_informative": False,
        "canonical_label": "not_informative",
        "trauma_status": "not_informative",
        "body_system": "not_informative",
        "symptom_type": "not_informative",
        "acuity_1to5": 0,
        "severity_1to5": 0,
        "specificity_1to5": 0,
        "red_flag": False,
        "objective_finding": False,
        "note": "Abbreviation PTA is too ambiguous in this fragment.",
    },
    {
        "normalized_component": "mulitple lacs",
        "is_informative": True,
        "canonical_label": "multiple lacerations",
        "trauma_status": "trauma",
        "body_system": "musculoskeletal",
        "symptom_type": "injury",
        "acuity_1to5": 4,
        "severity_1to5": 3,
        "specificity_1to5": 4,
        "red_flag": False,
        "objective_finding": True,
        "note": "Misspelled shorthand for multiple lacerations.",
    },
    {
        "normalized_component": "s.b.o. on ct",
        "is_informative": True,
        "canonical_label": "small bowel obstruction on ct",
        "trauma_status": "non_trauma",
        "body_system": "gastrointestinal",
        "symptom_type": "symptom",
        "acuity_1to5": 4,
        "severity_1to5": 4,
        "specificity_1to5": 5,
        "red_flag": True,
        "objective_finding": True,
        "note": "Imaging finding consistent with small bowel obstruction.",
    },
    {
        "normalized_component": "s.t./cough",
        "is_informative": True,
        "canonical_label": "sore throat and cough",
        "trauma_status": "non_trauma",
        "body_system": "respiratory",
        "symptom_type": "symptom",
        "acuity_1to5": 2,
        "severity_1to5": 2,
        "specificity_1to5": 4,
        "red_flag": False,
        "objective_finding": False,
        "note": "Common upper respiratory symptom combination.",
    },
    {
        "normalized_component": "soc",
        "is_informative": False,
        "canonical_label": "not_informative",
        "trauma_status": "not_informative",
        "body_system": "not_informative",
        "symptom_type": "not_informative",
        "acuity_1to5": 0,
        "severity_1to5": 0,
        "specificity_1to5": 0,
        "red_flag": False,
        "objective_finding": False,
        "note": "Too ambiguous as a standalone abbreviation.",
    },
    {
        "normalized_component": "sscp/dizziness",
        "is_informative": True,
        "canonical_label": "substernal chest pain and dizziness",
        "trauma_status": "non_trauma",
        "body_system": "cardiac",
        "symptom_type": "symptom",
        "acuity_1to5": 4,
        "severity_1to5": 4,
        "specificity_1to5": 4,
        "red_flag": True,
        "objective_finding": False,
        "note": "SSCP likely stands for substernal chest pain.",
    },
    {
        "normalized_component": "status post palps",
        "is_informative": True,
        "canonical_label": "palpitations",
        "trauma_status": "non_trauma",
        "body_system": "cardiac",
        "symptom_type": "symptom",
        "acuity_1to5": 3,
        "severity_1to5": 2,
        "specificity_1to5": 3,
        "red_flag": False,
        "objective_finding": False,
        "note": "Contextual prefix ignored; palps likely means palpitations.",
    },
    {
        "normalized_component": "status post sz ^ammonia level",
        "is_informative": True,
        "canonical_label": "seizure",
        "trauma_status": "non_trauma",
        "body_system": "neurologic",
        "symptom_type": "symptom",
        "acuity_1to5": 5,
        "severity_1to5": 4,
        "specificity_1to5": 4,
        "red_flag": True,
        "objective_finding": True,
        "note": "Post-seizure evaluation context; sz likely means seizure.",
    },
]

patch_map = {rec["normalized_component"]: rec for rec in patch_records}

cols = set(component_table.columns)

if "normalized_component" not in cols:
    raise KeyError("component_table 缺少 normalized_component")

component_col = "component" if "component" in cols else ("component_example" if "component_example" in cols else None)
if component_col is None:
    raise KeyError(f"component_table 缺少 component / component_example。現在欄位：{sorted(cols)}")

examples_col = "component_examples" if "component_examples" in cols else ("raw_examples_top5" if "raw_examples_top5" in cols else None)

need_count_cols = ["n_total", "n_datasets_present", "n_radiology", "n_pressor", "n_vent"]
missing_counts = [c for c in need_count_cols if c not in cols]
if missing_counts:
    raise KeyError(f"component_table 缺少計數欄位：{missing_counts}。現在欄位：{sorted(cols)}")

meta_df = component_table[
    ["normalized_component", component_col] + ([examples_col] if examples_col is not None else []) + need_count_cols
].drop_duplicates("normalized_component").copy()

meta_df = meta_df.rename(columns={component_col: "component"})

meta_df["component"] = meta_df["component"].astype(str).str.strip()

if examples_col == "component_examples":
    meta_df["component_examples"] = meta_df["component_examples"].apply(
        lambda x: list(dict.fromkeys([str(v).strip() for v in x if str(v).strip()]))
        if isinstance(x, list)
        else ([str(x).strip()] if pd.notna(x) and str(x).strip() else [])
    )
elif examples_col == "raw_examples_top5":
    meta_df["component_examples"] = meta_df["raw_examples_top5"].apply(
        lambda x: list(dict.fromkeys(
            [s.strip() for s in str(x).split("||") if s.strip()]
        )) if pd.notna(x) and str(x).strip() else []
    )
    meta_df["component_examples"] = meta_df.apply(
        lambda r: r["component_examples"] if len(r["component_examples"]) > 0 else [r["component"]],
        axis=1,
    )
    meta_df = meta_df.drop(columns=["raw_examples_top5"])
else:
    meta_df["component_examples"] = meta_df["component"].apply(lambda s: [s] if str(s).strip() else [])

meta_df["component_examples"] = meta_df["component_examples"].apply(
    lambda xs: list(dict.fromkeys([str(v).strip() for v in xs if str(v).strip()]))
)

meta_map = meta_df.set_index("normalized_component").to_dict(orient="index")

decoder = JSONDecoder()

main_text = output_path.read_text(encoding="utf-8") if output_path.exists() else ""
main_records = []
i = 0
main_malformed = 0
while i < len(main_text):
    while i < len(main_text) and main_text[i].isspace():
        i += 1
    if i >= len(main_text):
        break
    try:
        obj, j = decoder.raw_decode(main_text, i)
        main_records.append(obj)
        i = j
    except json.JSONDecodeError:
        i += 1
        main_malformed += 1

final_map = {}

for rec in main_records:
    if not isinstance(rec, dict):
        continue

    key = str(rec.get("normalized_component", "")).strip()
    if not key or key not in meta_map:
        continue

    meta = meta_map[key]
    src = patch_map[key] if key in patch_map else rec

    final_map[key] = {
        "normalized_component": key,
        "component": str(meta["component"]).strip(),
        "component_examples": meta["component_examples"],
        "n_total": int(meta["n_total"]),
        "n_datasets_present": int(meta["n_datasets_present"]),
        "n_radiology": int(meta["n_radiology"]),
        "n_pressor": int(meta["n_pressor"]),
        "n_vent": int(meta["n_vent"]),
        "is_informative": bool(src.get("is_informative", False)),
        "canonical_label": str(src.get("canonical_label", "")).strip(),
        "trauma_status": str(src.get("trauma_status", "")).strip(),
        "body_system": str(src.get("body_system", "")).strip(),
        "symptom_type": str(src.get("symptom_type", "")).strip(),
        "acuity_1to5": int(src.get("acuity_1to5", 0)),
        "severity_1to5": int(src.get("severity_1to5", 0)),
        "specificity_1to5": int(src.get("specificity_1to5", 0)),
        "red_flag": bool(src.get("red_flag", False)),
        "objective_finding": bool(src.get("objective_finding", False)),
        "note": str(src.get("note", "")).strip(),
    }

for key, src in patch_map.items():
    if key not in meta_map:
        print(f"WARNING: patch key not found in component_table -> {key}")
        continue

    meta = meta_map[key]
    final_map[key] = {
        "normalized_component": key,
        "component": str(meta["component"]).strip(),
        "component_examples": meta["component_examples"],
        "n_total": int(meta["n_total"]),
        "n_datasets_present": int(meta["n_datasets_present"]),
        "n_radiology": int(meta["n_radiology"]),
        "n_pressor": int(meta["n_pressor"]),
        "n_vent": int(meta["n_vent"]),
        "is_informative": bool(src["is_informative"]),
        "canonical_label": str(src["canonical_label"]).strip(),
        "trauma_status": str(src["trauma_status"]).strip(),
        "body_system": str(src["body_system"]).strip(),
        "symptom_type": str(src["symptom_type"]).strip(),
        "acuity_1to5": int(src["acuity_1to5"]),
        "severity_1to5": int(src["severity_1to5"]),
        "specificity_1to5": int(src["specificity_1to5"]),
        "red_flag": bool(src["red_flag"]),
        "objective_finding": bool(src["objective_finding"]),
        "note": str(src["note"]).strip(),
    }

fixed_df = pd.DataFrame(final_map.values(), columns=target_cols).sort_values(
    ["n_total", "normalized_component"],
    ascending=[False, True],
).reset_index(drop=True)

if output_path.exists():
    backup_main.write_text(output_path.read_text(encoding="utf-8"), encoding="utf-8")

with output_path.open("w", encoding="utf-8") as f:
    for rec in fixed_df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

error_text = error_path.read_text(encoding="utf-8") if error_path.exists() else ""
error_records = []
i = 0
err_malformed = 0
while i < len(error_text):
    while i < len(error_text) and error_text[i].isspace():
        i += 1
    if i >= len(error_text):
        break
    try:
        obj, j = decoder.raw_decode(error_text, i)
        error_records.append(obj)
        i = j
    except json.JSONDecodeError:
        i += 1
        err_malformed += 1

resolved_keys = set(fixed_df["normalized_component"].astype(str))
clean_error_records = []
seen_error_keys = set()

for rec in error_records:
    if not isinstance(rec, dict):
        continue
    key = str(rec.get("normalized_component", "")).strip()
    if not key or key in resolved_keys or key in seen_error_keys:
        continue
    seen_error_keys.add(key)
    clean_error_records.append(rec)

if error_path.exists():
    backup_err.write_text(error_path.read_text(encoding="utf-8"), encoding="utf-8")

with error_path.open("w", encoding="utf-8") as f:
    for rec in clean_error_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Main malformed fragments skipped: {main_malformed}")
print(f"Error malformed fragments skipped: {err_malformed}")
print(f"Rewrote main jsonl to: {output_path}")
print(f"Rewrote error jsonl to: {error_path}")
print(f"Backup main: {backup_main}")
print(f"Backup err: {backup_err}")
print(f"Rows in fixed main jsonl: {len(fixed_df)}")
print(f"Rows remaining in error jsonl: {len(clean_error_records)}")

display(
    fixed_df.loc[
        fixed_df["normalized_component"].isin(patch_map.keys()),
        target_cols
    ].sort_values("normalized_component").reset_index(drop=True)
)

Main malformed fragments skipped: 0
Error malformed fragments skipped: 0
Rewrote main jsonl to: main/chief_complaint_lookup_table.jsonl
Rewrote error jsonl to: main/chief_complaint_lookup_table_errors.jsonl
Backup main: main/chief_complaint_lookup_table_backup_before_manual_schema_fix.jsonl
Backup err: main/chief_complaint_lookup_table_errors_backup_before_manual_schema_fix.jsonl
Rows in fixed main jsonl: 13958
Rows remaining in error jsonl: 0


,normalized_component,component,component_examples,n_total,n_datasets_present,n_radiology,n_pressor,n_vent,is_informative,canonical_label,trauma_status,body_system,symptom_type,acuity_1to5,severity_1to5,specificity_1to5,red_flag,objective_finding,note
0,b rib pain,B Rib pain,[B Rib pain],1,1,1,0,0,True,rib pain,non_trauma,musculoskeletal,symptom,3,2,3,False,False,Likely rib pain; leading abbreviation is ambig...
1,boerrhaves,BOERRHAVES,[BOERRHAVES],1,1,1,0,0,True,boerhaave syndrome,non_trauma,gastrointestinal,symptom,5,5,5,True,True,Likely misspelling of Boerhaave syndrome.
2,"dt""s si","DT""S SI","[DT""S SI]",1,1,1,0,0,False,not_informative,not_informative,not_informative,not_informative,0,0,0,False,False,Ambiguous abbreviation string; not clinically ...
3,eu critial/motor vehicle collision,EU CRITIAL/MVC,[EU CRITIAL/MVC],1,1,1,0,0,True,motor vehicle collision trauma,trauma,musculoskeletal,injury,4,4,4,True,False,Trauma mechanism is clear from motor vehicle c...
4,ich-hip pain,ICH-HIP PAIN,[ICH-HIP PAIN],1,1,1,0,0,True,hip pain,non_trauma,musculoskeletal,symptom,3,2,3,False,False,Hip pain is interpretable; leading ICH prefix ...
5,iph,IPH,[IPH],112,3,65,36,11,True,intraparenchymal hemorrhage,non_trauma,neurologic,symptom,5,5,4,True,True,Likely chief complaint abbreviation for intrap...
6,left pta,LEFT PTA,[LEFT PTA],1,1,1,0,0,False,not_informative,not_informative,not_informative,not_informative,0,0,0,False,False,Abbreviation PTA is too ambiguous in this frag...
7,mulitple lacs,MULITPLE LACS,[MULITPLE LACS],1,1,1,0,0,True,multiple lacerations,trauma,musculoskeletal,injury,4,3,4,False,True,Misspelled shorthand for multiple lacerations.
8,s.b.o. on ct,S.B.O. ON CT,[S.B.O. ON CT],1,1,1,0,0,True,small bowel obstruction on ct,non_trauma,gastrointestinal,symptom,4,4,5,True,True,Imaging finding consistent with small bowel ob...
9,s.t./cough,S.T. /COUGH,[S.T. /COUGH],1,1,1,0,0,True,sore throat and cough,non_trauma,respiratory,symptom,2,2,4,False,False,Common upper respiratory symptom combination.
